# F1 Race Result Predictor — Model Evaluation

Loads the trained RFR and XGB models from `train.ipynb` and evaluates them  
against the held-out **2025 season** data.

Evaluation is done at two levels:
1. **Season-level** — average metrics across all 2025 races
2. **Race-level** — per-race Spearman correlation breakdown
3. **Single-race deep-dive** — full driver-by-driver prediction table

Metrics used: MAE, RMSE, R², Median AE, Spearman ρ, Kendall τ

## 1. Imports

In [ ]:
import glob
import pickle
import os

import pandas as pd
from scipy.stats import kendalltau, spearmanr
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
    root_mean_squared_error,
)

from names_and_ids import driver_names

## 2. Load Models and Test Data

In [ ]:
with open("../models/best_models/best_rfr_model.pkl", "rb") as f:
    rfr_model = pickle.load(f)

with open("../models/best_models/best_xgb_model.pkl", "rb") as f:
    xgb_model = pickle.load(f)

print("Models loaded.")

Models loaded.


In [ ]:
file_list_2025 = glob.glob("../data/2025/*.csv")
print(f"Found {len(file_list_2025)} race files for 2025")

Found 24 race files for 2025


## 3. Helper Functions

In [9]:
def load_race(filepath):
    """Load a single race CSV, drop DNFs and rows with missing finishing position."""
    df = pd.read_csv(filepath)
    df = df.dropna(subset=["FinishingPos"])
    df = df[df["Retired"] == False]
    X = df.drop(["FinishingPos", "Retired"], axis=1)
    y = df["FinishingPos"]
    return X, y


def get_predicted_ranks(X, Y_pred):
    """Convert raw model output values to integer finishing position ranks per race."""
    df = pd.DataFrame({
        "RaceId":         X["RacesInGEEra"],
        "PredictedValue": Y_pred,
    })
    df["PredictedRank"] = (
        df.groupby("RaceId")["PredictedValue"]
        .rank(method="first", ascending=True)
        .astype(int)
    )
    return df


def compute_metrics(y_true, y_pred_ranks):
    """Return a dict of regression and ranking metrics."""
    return {
        "MAE":      mean_absolute_error(y_true, y_pred_ranks),
        "MSE":      mean_squared_error(y_true, y_pred_ranks),
        "RMSE":     root_mean_squared_error(y_true, y_pred_ranks),
        "R2":       r2_score(y_true, y_pred_ranks),
        "MedianAE": median_absolute_error(y_true, y_pred_ranks),
        "Spearman": spearmanr(y_true, y_pred_ranks)[0],
        "Kendall":  kendalltau(y_true, y_pred_ranks)[0],
    }

## 4. Season-Level Evaluation

Average metrics across all 2025 races for both models.

In [10]:
results = {"RFR": [], "XGB": []}

for file in file_list_2025:
    X, y = load_race(file)
    for name, model in [("RFR", rfr_model), ("XGB", xgb_model)]:
        preds = model.predict(X)
        ranks = get_predicted_ranks(X, preds)["PredictedRank"]
        results[name].append(compute_metrics(y, ranks))

for model_name, race_metrics in results.items():
    avg = pd.DataFrame(race_metrics).mean()
    print(f"\n--- {model_name} (2025 season average) ---")
    for metric, value in avg.items():
        print(f"  {metric}: {value:.4f}")


--- RFR (2025 season average) ---
  MAE: 2.2083
  MSE: 9.8648
  RMSE: 3.0146
  R2: 0.5786
  MedianAE: 1.4583
  Spearman: 0.7893
  Kendall: 0.6369

--- XGB (2025 season average) ---
  MAE: 1.9961
  MSE: 8.9306
  RMSE: 2.8377
  R2: 0.6217
  MedianAE: 1.4583
  Spearman: 0.8108
  Kendall: 0.7204


## 5. Per-Race Spearman Breakdown

Spearman correlation for each race in the 2025 season (XGB model).

In [11]:
print(f"{'Race':<30} {'Spearman ρ':>10}")
print("-" * 42)

for file in sorted(file_list_2025):
    X, y = load_race(file)
    preds = xgb_model.predict(X)
    ranks = get_predicted_ranks(X, preds)["PredictedRank"]
    rho = spearmanr(y, ranks)[0]
    race_name = file.replace("2025/", "").replace("2025\\\\", "").replace("_race.csv", "")
    print(f"{race_name:<30} {rho:>10.4f}")

Race                           Spearman ρ
------------------------------------------
2025\Austin                        0.9368
2025\Baku                          0.9719
2025\Barcelona                     0.7721
2025\Budapest                      0.8211
2025\Imola                         0.7977
2025\Jeddah                        0.9690
2025\Las Vegas                     0.6107
2025\Lusail                        0.8559
2025\Marina Bay                    0.9549
2025\Melbourne                     0.7011
2025\Mexico City                   0.8206
2025\Miami                         0.9029
2025\Monaco                        0.7234
2025\Montreal                      0.8431
2025\Monza                         0.9216
2025\Sakhir                        0.9051
2025\Sao Paulo                     0.8554
2025\Shanghai                      0.7235
2025\Silverstone                   0.3679
2025\Spa-Francorchamps             0.7699
2025\Spielberg                     0.7441
2025\Suzuka                      

## 6. Single-Race Deep Dive

Full driver-level breakdown for a specific race.  
Change `RACE_FILE` to inspect any round.

In [ ]:
RACE_FILE = "../2025/Suzuka_race.csv"

X_race, y_race = load_race(RACE_FILE)
preds = xgb_model.predict(X_race)
df_preds = get_predicted_ranks(X_race, preds)

metrics = compute_metrics(y_race, df_preds["PredictedRank"])
print(f"Metrics for {RACE_FILE}:")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.4f}")

Metrics for 2025/Suzuka_race.csv:
  MAE: 0.6000
  MSE: 1.6000
  RMSE: 1.2649
  R2: 0.9519
  MedianAE: 0.0000
  Spearman: 0.9759
  Kendall: 0.9368


In [13]:
print(f"{'Driver':<25} {'Quali':>6} {'Predicted':>10} {'Actual':>8}")
print("-" * 52)

for i, pred_row in df_preds.iterrows():
    driver  = driver_names[X_race["DriverId"][i]]
    quali   = X_race["QualifyingPos"][i]
    predicted = pred_row["PredictedRank"]
    actual  = y_race[i]
    print(f"{driver:<25} {quali:>6} {predicted:>10} {actual:>8}")

Driver                     Quali  Predicted   Actual
----------------------------------------------------
Max Verstappen               1.0        1.0      1.0
Lando Norris                 2.0        2.0      2.0
Oscar Piastri                3.0        3.0      3.0
Charles Leclerc              4.0        5.0      4.0
George Russell               5.0        4.0      5.0
Kimi Antonelli               6.0        6.0      6.0
Lewis Hamilton               8.0        8.0      7.0
Isack Hadjar                 7.0        9.0      8.0
Alexander Albon              9.0       10.0      9.0
Oliver Bearman              10.0       11.0     10.0
Fernando Alonso             13.0       12.0     11.0
Yuki Tsunoda                15.0        7.0     12.0
Pierre Gasly                11.0       13.0     13.0
Carlos Sainz                12.0       14.0     14.0
Jack Doohan                 19.0       15.0     15.0
Nico Hulkenberg             16.0       16.0     16.0
Liam Lawson                 14.0       17.0   

## 7. Saving Results to Disk

Writes all evaluation outputs to the `results/` directory for reference.

- `results/{model}_per_race.csv` — all metrics for each 2025 race
- `results/{model}_season_avg.csv` — season average metrics
- `results/predictions/{model}/2025/{race}.csv` — driver-level predictions for each race (qualifying position, predicted finish, actual finish)

In [ ]:
os.makedirs("../results/predictions/xgb", exist_ok=True)
os.makedirs("../results/predictions/rfr", exist_ok=True)

for model_name, model in [("rfr", rfr_model), ("xgb", xgb_model)]:
    per_race_rows = []

    for file in sorted(file_list_2025):
        X, y = load_race(file)
        preds = model.predict(X)
        df_preds = get_predicted_ranks(X, preds)
        metrics = compute_metrics(y, df_preds["PredictedRank"])
        race_name = file.replace("2025/", "").replace("2025\\\\", "").replace("_race.csv", "")

        per_race_rows.append({"Race": race_name, **metrics})

        prediction_df = pd.DataFrame({
            "Driver":          [driver_names[X["DriverId"][i]] for i in X.index],
            "QualifyingPos":   X["QualifyingPos"].values,
            "PredictedFinish": df_preds["PredictedRank"].values,
            "ActualFinish":    y.values,
        }).sort_values("ActualFinish")

        prediction_df.to_csv(f"../results/predictions/{model_name}/{race_name}.csv", index=False)

    per_race_df = pd.DataFrame(per_race_rows)
    per_race_df.to_csv(f"../results/{model_name}_per_race.csv", index=False)

    avg_row = per_race_df.drop(columns="Race").mean().to_frame(name="Average").T
    avg_row.to_csv(f"../results/{model_name}_season_avg.csv", index=False)

print("Results saved to results/")

Results saved to results/
